# Deliverable 3: Classification, Clustering, and Pattern Mining

**Course Project:** Advanced Data Mining for Data-Driven Insights and Predictive Modeling

This notebook completes Deliverable 3 by building classification and clustering models, applying association rule mining, tuning hyperparameters, and interpreting actionable insights.

## 1) Imports and Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Classification and evaluation
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve, silhouette_score
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Clustering
from sklearn.cluster import KMeans

# Pattern mining
from mlxtend.frequent_patterns import apriori, association_rules

# Display and plot settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid', context='notebook')

## 2) Load Data and Define Targets

For classification, we create a binary target from `test_results`:
- `1` if result is `Abnormal`
- `0` otherwise

In [ ]:
# Load cleaned data from Deliverable 1
file_path = 'healthcare_dataset_cleaned.csv'
df = pd.read_csv(file_path)

# Basic checks
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

# Binary target for classification
df['abnormal_flag'] = (df['test_results'] == 'Abnormal').astype(int)

print('Class distribution (abnormal_flag):')
print(df['abnormal_flag'].value_counts(normalize=True).round(4))

df.head()

## 3) Classification Modeling

We build two classification models:
1. Decision Tree
2. k-Nearest Neighbors (k-NN)

Then we tune Decision Tree hyperparameters using GridSearchCV.

In [ ]:
# Prepare features and target for classification
# We exclude direct identifiers and raw date columns for more generalizable models.
X = df.drop(columns=['abnormal_flag', 'test_results', 'name', 'doctor', 'hospital', 'date_of_admission', 'discharge_date'])
y = df['abnormal_flag']

categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print('Numeric features:', numeric_cols)
print('Categorical features:', categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

In [ ]:
# Shared preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'k-NN': KNeighborsClassifier(n_neighbors=7)
}

classification_results = []
trained_classifiers = {}

for model_name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    classification_results.append({
        'Model': model_name,
        'Accuracy': acc,
        'F1 Score': f1,
        'ROC AUC': auc
    })

    trained_classifiers[model_name] = {
        'pipeline': pipe,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

classification_results_df = pd.DataFrame(classification_results).sort_values('F1 Score', ascending=False).reset_index(drop=True)
classification_results_df

In [ ]:
# Hyperparameter tuning for Decision Tree
# Tuning objective: maximize F1 score on cross-validation splits.
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

param_grid = {
    'model__criterion': ['gini', 'entropy'],
    'model__max_depth': [3, 5, 8, 12, None],
    'model__min_samples_split': [2, 10, 30],
    'model__min_samples_leaf': [1, 5, 15]
}

grid_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_dt = grid_search.best_estimator_

best_y_pred = best_dt.predict(X_test)
best_y_proba = best_dt.predict_proba(X_test)[:, 1]

tuned_accuracy = accuracy_score(y_test, best_y_pred)
tuned_f1 = f1_score(y_test, best_y_pred)
tuned_auc = roc_auc_score(y_test, best_y_proba)

print('Best Parameters:', grid_search.best_params_)
print('Best CV F1:', round(grid_search.best_score_, 4))
print('Tuned Test Accuracy:', round(tuned_accuracy, 4))
print('Tuned Test F1:', round(tuned_f1, 4))
print('Tuned Test ROC AUC:', round(tuned_auc, 4))

In [ ]:
# Add tuned model results to the comparison table
classification_results_df = pd.concat([
    classification_results_df,
    pd.DataFrame([{
        'Model': 'Decision Tree (Tuned)',
        'Accuracy': tuned_accuracy,
        'F1 Score': tuned_f1,
        'ROC AUC': tuned_auc
    }])
], ignore_index=True).sort_values('F1 Score', ascending=False).reset_index(drop=True)

classification_results_df.round(4)

## 4) Classification Evaluation

Required metrics included below:
- Confusion matrix
- ROC curve
- Accuracy and F1 score

In [ ]:
# Confusion matrices for base models and tuned decision tree
model_plot_data = {
    'Decision Tree': trained_classifiers['Decision Tree'],
    'k-NN': trained_classifiers['k-NN'],
    'Decision Tree (Tuned)': {'y_pred': best_y_pred, 'y_proba': best_y_proba}
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, data_dict) in zip(axes, model_plot_data.items()):
    cm = confusion_matrix(y_test, data_dict['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(f'Confusion Matrix\n{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
plt.figure(figsize=(8, 6))

for name, data_dict in model_plot_data.items():
    fpr, tpr, _ = roc_curve(y_test, data_dict['y_proba'])
    auc = roc_auc_score(y_test, data_dict['y_proba'])
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.title('ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model metric comparison plot
metrics_long = classification_results_df.melt(
    id_vars='Model',
    value_vars=['Accuracy', 'F1 Score', 'ROC AUC'],
    var_name='Metric',
    value_name='Value'
)

plt.figure(figsize=(10, 5))
sns.barplot(data=metrics_long, x='Model', y='Value', hue='Metric')
plt.title('Classification Metric Comparison')
plt.xlabel('Model')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

classification_results_df.round(4)

## 5) Clustering (K-Means)

We cluster patients using numeric features:
- `age`
- `billing_amount`
- `room_number`
- `length_of_stay`

We choose the number of clusters based on silhouette score across candidate values of $k$.

In [ ]:
# Prepare data for clustering
cluster_features = ['age', 'billing_amount', 'room_number', 'length_of_stay']
cluster_df = df[cluster_features].copy()

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(cluster_df)

silhouette_scores = {}
for k in range(2, 7):
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_temp = kmeans_temp.fit_predict(X_cluster_scaled)
    silhouette_scores[k] = silhouette_score(X_cluster_scaled, labels_temp)

silhouette_df = pd.DataFrame({
    'k': list(silhouette_scores.keys()),
    'silhouette_score': list(silhouette_scores.values())
})

best_k = max(silhouette_scores, key=silhouette_scores.get)
print('Best k by silhouette score:', best_k)
silhouette_df

In [ ]:
# Silhouette trend plot
plt.figure(figsize=(7, 4))
sns.lineplot(data=silhouette_df, x='k', y='silhouette_score', marker='o')
plt.title('Silhouette Score by Number of Clusters (k)')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette score')
plt.xticks(silhouette_df['k'])
plt.tight_layout()
plt.show()

In [ ]:
# Fit final K-Means with best k
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster_scaled)

cluster_output = cluster_df.copy()
cluster_output['cluster'] = cluster_labels

# Cluster profile table
cluster_profile = cluster_output.groupby('cluster')[cluster_features].mean().round(2)
cluster_counts = cluster_output['cluster'].value_counts().sort_index()

print('Cluster sizes:')
print(cluster_counts)

cluster_profile

In [ ]:
# Visualize clusters in 2D using PCA projection
pca = PCA(n_components=2, random_state=42)
pca_points = pca.fit_transform(X_cluster_scaled)

pca_df = pd.DataFrame({
    'PC1': pca_points[:, 0],
    'PC2': pca_points[:, 1],
    'cluster': cluster_labels.astype(str)
})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='cluster', alpha=0.55, s=35)
plt.title('K-Means Clusters Visualized with PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

## 6) Association Rule Mining (Apriori)

We mine frequent patterns using selected categorical columns:
- medical condition
- medication
- admission type
- insurance provider
- test results
- gender
- blood type

Then we generate rules with meaningful confidence and lift.

In [ ]:
# Create one-hot encoded basket format for Apriori
basket_cols = [
    'medical_condition', 'medication', 'admission_type',
    'insurance_provider', 'test_results', 'gender', 'blood_type'
]

basket = pd.get_dummies(df[basket_cols].astype(str), prefix=basket_cols)

# Frequent itemsets and rules
frequent_itemsets = apriori(basket, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)

# Keep stronger rules for interpretation
rules_filtered = rules[(rules['confidence'] >= 0.30) & (rules['lift'] > 1.0)].copy()
rules_filtered = rules_filtered.sort_values(['lift', 'confidence'], ascending=False)

print('Frequent itemsets shape:', frequent_itemsets.shape)
print('Filtered rules shape:', rules_filtered.shape)

rules_filtered[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

In [ ]:
# Visualize top rules by lift
top_rules = rules_filtered[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).copy()
top_rules['rule'] = top_rules['antecedents'].astype(str) + ' -> ' + top_rules['consequents'].astype(str)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_rules, y='rule', x='lift', hue='rule', legend=False, palette='crest')
plt.title('Top 10 Association Rules by Lift')
plt.xlabel('Lift')
plt.ylabel('Rule')
plt.tight_layout()
plt.show()

top_rules

## 7) Interpretation and Real-World Application

### Classification Insights
- Decision Tree and k-NN provide a baseline for predicting abnormal test outcomes.
- Hyperparameter tuning was performed with GridSearchCV on Decision Tree parameters (`criterion`, `max_depth`, `min_samples_split`, `min_samples_leaf`) using F1 as the optimization metric.
- Confusion matrix and ROC analysis reveal trade-offs between overall accuracy and minority-class detection quality.

### Clustering Insights
- K-Means identified naturally grouped patient segments using cost, stay length, age, and room assignment patterns.
- These segments can support operational decisions like resource planning, admission triage support, and service-level optimization.

### Pattern Mining Insights
- Association rules reveal recurring co-occurrences among medical condition, admission type, medication, and outcome categories.
- In practice, these rules can inform care pathway design, medication inventory planning, and risk-aware monitoring strategies.

### Deliverable 3 Conclusion
This deliverable integrates supervised learning, unsupervised learning, and rule-based pattern discovery in one pipeline. Together, these methods provide a broader view of patient outcomes, population structure, and actionable healthcare patterns for decision support.